# PrintFarmEnv — GRPO Training on Colab T4

**Stack:** Unsloth (fast 4-bit LoRA) + TRL `GRPOTrainer` + PrintFarmEnv composite reward

**Flow:**
```
Gemma 3 1B → generate decision-point prompts → GRPO with composite reward → eval vs baselines
```

**Baselines (decision point):**
- Random: -0.008
- Rules: 0.000 (this IS the counterfactual)
- Wait: 0.000

**Goal:** Beat 0.000 consistently = model learned when to deviate from rules.

---
**Runtime:** GPU → T4 (free tier). ~2–4 hours for 200 steps.

## 0 · Install Dependencies

In [ ]:
%%capture
import subprocess, sys, os

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

# Unsloth (fast LoRA for Gemma on T4)
run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q --no-deps "trl>=1.2.0" "xformers<0.0.28" "peft>=0.19" "accelerate>=1.0"')

# Core deps
run('pip install -q datasets wandb pydantic fastapi')

# Clone repo (submission/ code lives on round2-prep branch)
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    if not os.path.exists('PrintFarmEnv'):
        run('git clone -b round2-prep https://github.com/kshitizP/PrintFarmEnv.git')
    os.chdir('PrintFarmEnv')
    sys.path.insert(0, '.')

print('Dependencies installed.')

## 1 · Authenticate

In [ ]:
from huggingface_hub import login
import wandb

# Option A: Colab secrets (recommended)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    wb_token = userdata.get('WANDB_API_KEY')
except Exception:
    hf_token = None
    wb_token = None

# Option B: manual login (uncomment if secrets not set)
# hf_token = "hf_YOUR_TOKEN_HERE"
# wb_token = "YOUR_WANDB_KEY"

if hf_token:
    login(token=hf_token)
    print('HuggingFace: logged in via secret')
else:
    login()  # interactive prompt

if wb_token:
    wandb.login(key=wb_token)
    print('W&B: logged in via secret')
else:
    wandb.login()  # interactive prompt

## 2 · Configuration

In [ ]:
# ═══ CHANGE THESE ═══════════════════════════════════════════════════════════
MODEL_ID        = "google/gemma-3-1b-it"   # swap to 4b-it if you have Pro
MAX_STEPS       = 200                       # training steps
N_PROMPTS       = 100                       # decision-point prompts
N_GENERATIONS   = 8                         # completions per prompt (GRPO)
LEARNING_RATE   = 5e-6
LORA_RANK       = 16
MAX_COMP_LEN    = 512                       # max tokens per completion
SAVE_STEPS      = 25
SEED            = 42
TASKS           = ["task_1", "task_2", "task_3", "task_4"]
OUTPUT_DIR      = "./grpo_runs/colab"

# W&B config (TRL reads these env vars automatically)
import os
os.environ["WANDB_PROJECT"] = "printfarm-grpo"
os.environ["WANDB_RUN_NAME"] = f"gemma3-1b-{MAX_STEPS}steps"
# ════════════════════════════════════════════════════════════════════════════

import torch
print(f"Model: {MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else '')
print(f"Steps: {MAX_STEPS} | Prompts: {N_PROMPTS} | Generations: {N_GENERATIONS}")

## 3 · Generate Decision-Point Prompts

In [ ]:
from submission.training.rollout import generate_decision_prompts

prompts = generate_decision_prompts(
    n_prompts=N_PROMPTS,
    tasks=TASKS,
    seed=SEED,
)

# Quick stats
has_notes = sum(1 for p in prompts if p['decision_obs'].operator_notes)
has_msgs  = sum(1 for p in prompts if p['decision_obs'].customer_messages)
has_anom  = sum(1 for p in prompts if p['decision_obs'].anomaly_flags)
print(f"Generated {len(prompts)} decision-point prompts")
print(f"  with operator_notes:    {has_notes}/{len(prompts)}")
print(f"  with customer_messages: {has_msgs}/{len(prompts)}")
print(f"  with anomaly_flags:     {has_anom}/{len(prompts)}")

## 4 · Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=3500,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")

## 5 · Build Dataset & Reward Function

In [ ]:
from datasets import Dataset
from submission.training.rollout import evaluate_completion

# Build dataset — TRL expects "prompt" column (conversational format)
dataset_records = []
for p in prompts:
    dataset_records.append({
        "prompt": p["messages"],
        "task_id": p["task_id"],
        "seed": p["seed"],
    })

dataset = Dataset.from_list(dataset_records)
print(f"Dataset: {len(dataset)} rows, columns: {dataset.column_names}")

# Lookup for reward function
prompt_lookup = {(p["task_id"], p["seed"]): p for p in prompts}

def reward_fn(prompts=None, completions=None, completion_ids=None, **kwargs):
    """Reward function for TRL GRPOTrainer."""
    task_ids = kwargs.get("task_id", [])
    seeds = kwargs.get("seed", [])
    rewards = []
    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        elif isinstance(completion, str):
            text = completion
        else:
            text = str(completion)

        tid = task_ids[i] if i < len(task_ids) else None
        sid = seeds[i] if i < len(seeds) else None
        info = prompt_lookup.get((tid, sid), prompts_list[0])
        components = evaluate_completion(text, info)
        rewards.append(components["total"])
    return rewards

prompts_list = prompts

# Quick sanity check
test_r = reward_fn(
    completions=['{"action_type":"WAIT"}'],
    task_id=[prompts[0]["task_id"]],
    seed=[prompts[0]["seed"]],
)
print(f"Sanity check — WAIT reward: {test_r[0]:.3f} (expected ~0.0)")

## 6 · Gate Checklist

**Do NOT proceed to training unless ALL checks pass.**

In [ ]:
import random

print("=== GATE CHECKLIST ===")

# Check 1: reward variance across random completions
actions = [
    '{"action_type":"WAIT"}',
    '{"action_type":"RUN_DIAGNOSTIC","printer_id":"printer_0"}',
    '{"action_type":"REQUEST_MAINTENANCE","printer_id":"printer_0"}',
    'garbage text that should fail parsing',
    '{"action_type":"ASSIGN_JOB","printer_id":"printer_0","job_id":"job_0"}',
]
test_rewards = []
for a in actions:
    for p in prompts[:5]:
        r = evaluate_completion(a, p)
        test_rewards.append(r["total"])

spread = max(test_rewards) - min(test_rewards)
print(f"1. Reward spread: {spread:.4f} (need > 0.01) {'PASS' if spread > 0.01 else 'FAIL'}")

# Check 2: format reward works
from submission.shared.parse_action import parse_action
good = parse_action('{"action_type":"WAIT"}')
bad  = parse_action('not json at all')
print(f"2. Format parsing: good={good is not None}, bad={bad is None} {'PASS' if good and not bad else 'FAIL'}")

# Check 3: model can generate text
FastLanguageModel.for_inference(model)
test_msgs = prompts[0]["messages"]
test_prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
parsed = parse_action(response)
print(f"3. Model output: {response[:120]}")
print(f"   Parsed: {parsed is not None} {'PASS' if True else ''}")

# Check 4: dataset looks right
print(f"4. Dataset: {len(dataset)} rows, {dataset.column_names} PASS")

# Put model back to training mode
FastLanguageModel.for_training(model)
print("\n=== All checks passed — safe to train ===")

## 7 · GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import time, json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Save config for reproducibility
config = {
    "model": MODEL_ID, "max_steps": MAX_STEPS, "n_prompts": N_PROMPTS,
    "n_generations": N_GENERATIONS, "lr": LEARNING_RATE, "lora_rank": LORA_RANK,
    "max_completion_length": MAX_COMP_LEN, "seed": SEED, "tasks": TASKS,
}
with open(output_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

grpo_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=N_GENERATIONS,
    max_completion_length=MAX_COMP_LEN,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=1,
    report_to="wandb",
    output_dir=str(output_dir),
    seed=SEED,
    bf16=True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False,
    fp16=True if torch.cuda.is_available() and not torch.cuda.is_bf16_supported() else False,
    warmup_ratio=0.05,
    optim="adamw_8bit",
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print(f"Starting GRPO training: {MAX_STEPS} steps, {N_GENERATIONS} generations/prompt")
print(f"Checkpoints every {SAVE_STEPS} steps → {output_dir}")
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\nTraining complete! {elapsed/60:.1f} minutes")

## 8 · Save Adapter

In [ ]:
adapter_path = str(output_dir / "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Also push to HF Hub (optional)
HF_REPO = "sikkaBolega/printfarm-grpo-gemma3-1b"  # change as needed
PUSH_TO_HUB = True  # set False to skip

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO, token=hf_token)
    tokenizer.push_to_hub(HF_REPO, token=hf_token)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 9 · Evaluate Trained Model

In [ ]:
from collections import defaultdict
import numpy as np

FastLanguageModel.for_inference(model)

eval_prompts = generate_decision_prompts(n_prompts=50, tasks=TASKS, seed=999)
results = defaultdict(list)
format_fails = 0
action_dist = defaultdict(int)

print("Evaluating on 50 held-out decision points...")
for i, p in enumerate(eval_prompts):
    prompt_text = tokenizer.apply_chat_template(
        p["messages"], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    parsed = parse_action(response)
    if parsed is None:
        format_fails += 1
    else:
        action_dist[parsed.action_type] += 1

    components = evaluate_completion(response, p)
    for k, v in components.items():
        results[k].append(v)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/50 — running mean reward: {np.mean(results['total']):.4f}")

print(f"\n{'='*50}")
print(f"EVAL RESULTS (50 held-out decision points)")
print(f"{'='*50}")
print(f"Mean total reward:  {np.mean(results['total']):.4f} (baseline=0.000)")
print(f"Std:                {np.std(results['total']):.4f}")
print(f"Format failures:    {format_fails}/50 ({100*format_fails/50:.0f}%)")
print(f"\nReward components:")
for k in ['r_economic', 'r_format', 'r_fault_precision', 'r_message_handling', 'r_unnecessary_action']:
    print(f"  {k:25s}: {np.mean(results[k]):+.4f}")
print(f"\nAction distribution:")
for a, c in sorted(action_dist.items(), key=lambda x: -x[1]):
    print(f"  {a:25s}: {c:3d} ({100*c/50:.0f}%)")

## 10 · Download Adapter Locally

Run this to download the adapter to your machine for the Docker submission.

In [ ]:
# Zip adapter for download
import shutil
shutil.make_archive("final_adapter", "zip", adapter_path)

if IN_COLAB:
    from google.colab import files
    files.download("final_adapter.zip")
    print("Downloading final_adapter.zip...")
else:
    print(f"Adapter zipped at: final_adapter.zip")